# Hybrid QNN Testing on CICIoT2023

This notebook tests the trained Hybrid QNN model on the CICIoT2023 dataset.
It replicates the data loading and preprocessing steps to ensure the test set is consistent with the training phase.

In [ ]:
import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
import pennylane as qml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

## 1. Configuration & Data Loading (Must match Training)

In [ ]:
# Configuration
DATASET_DIRECTORY = 'Datasets/unb-cic-iot-datase/wataiData/csv/CICIoT2023/'
FILES_TO_LOAD = 5  # Must match training to get same split
BATCH_SIZE = 32
N_QUBITS = 4
N_LAYERS = 2
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# Define Class Mapping (Binary: Attack vs Benign)
dict_2classes = {}
dict_2classes['DDoS-RSTFINFlood'] = 'Attack'
dict_2classes['DDoS-PSHACK_Flood'] = 'Attack'
dict_2classes['DDoS-SYN_Flood'] = 'Attack'
dict_2classes['DDoS-UDP_Flood'] = 'Attack'
dict_2classes['DDoS-TCP_Flood'] = 'Attack'
dict_2classes['DDoS-ICMP_Flood'] = 'Attack'
dict_2classes['DDoS-SynonymousIP_Flood'] = 'Attack'
dict_2classes['DDoS-ACK_Fragmentation'] = 'Attack'
dict_2classes['DDoS-UDP_Fragmentation'] = 'Attack'
dict_2classes['DDoS-ICMP_Fragmentation'] = 'Attack'
dict_2classes['DDoS-SlowLoris'] = 'Attack'
dict_2classes['DDoS-HTTP_Flood'] = 'Attack'

dict_2classes['DoS-UDP_Flood'] = 'Attack'
dict_2classes['DoS-SYN_Flood'] = 'Attack'
dict_2classes['DoS-TCP_Flood'] = 'Attack'
dict_2classes['DoS-HTTP_Flood'] = 'Attack'

dict_2classes['Mirai-greeth_flood'] = 'Attack'
dict_2classes['Mirai-greip_flood'] = 'Attack'
dict_2classes['Mirai-udpplain'] = 'Attack'

dict_2classes['Recon-PingSweep'] = 'Attack'
dict_2classes['Recon-OSScan'] = 'Attack'
dict_2classes['Recon-PortScan'] = 'Attack'
dict_2classes['VulnerabilityScan'] = 'Attack'
dict_2classes['Recon-HostDiscovery'] = 'Attack'

dict_2classes['DNS_Spoofing'] = 'Attack'
dict_2classes['MITM-ArpSpoofing'] = 'Attack'

dict_2classes['BenignTraffic'] = 'Benign'

dict_2classes['BrowserHijacking'] = 'Attack'
dict_2classes['Backdoor_Malware'] = 'Attack'
dict_2classes['XSS'] = 'Attack'
dict_2classes['Uploading_Attack'] = 'Attack'
dict_2classes['SqlInjection'] = 'Attack'
dict_2classes['CommandInjection'] = 'Attack'

dict_2classes['DictionaryBruteForce'] = 'Attack'

In [ ]:
# Load Data
print("Loading dataset...")
csv_files = [f for f in os.listdir(DATASET_DIRECTORY) if f.endswith('.csv')]
csv_files.sort()
selected_files = csv_files[:FILES_TO_LOAD]

dfs = []
for file in tqdm(selected_files):
    path = os.path.join(DATASET_DIRECTORY, file)
    df = pd.read_csv(path)
    dfs.append(df)

full_df = pd.concat(dfs, ignore_index=True)
print(f"Total samples: {len(full_df)}")

In [ ]:
# Preprocessing
X_columns = [
    'flow_duration', 'Header_Length', 'Protocol Type', 'Duration',
    'Rate', 'Srate', 'Drate', 'fin_flag_number', 'syn_flag_number',
    'rst_flag_number', 'psh_flag_number', 'ack_flag_number',
    'ece_flag_number', 'cwr_flag_number', 'ack_count',
    'syn_count', 'fin_count', 'urg_count', 'rst_count', 
    'HTTP', 'HTTPS', 'DNS', 'Telnet', 'SMTP', 'SSH', 'IRC', 'TCP',
    'UDP', 'DHCP', 'ARP', 'ICMP', 'IPv', 'LLC', 'Tot sum', 'Min',
    'Max', 'AVG', 'Std', 'Tot size', 'IAT', 'Number', 'Magnitue',
    'Radius', 'Covariance', 'Variance', 'Weight', 
]
y_column = 'label'

# Map labels
full_df['binary_label'] = full_df[y_column].map(dict_2classes)
full_df['target'] = (full_df['binary_label'] == 'Attack').astype(int)

# Split Features and Target
X = full_df[X_columns].values
y = full_df['target'].values

# Scale Features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=SEED)

# Convert to PyTorch Tensors
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

# Create DataLoader
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Testing samples: {len(X_test)}")

## 2. Model Architecture & Loading

In [ ]:
# Quantum Device
dev = qml.device("default.qubit", wires=N_QUBITS)

@qml.qnode(dev, interface="torch")
def qnn_layer(inputs, weights):
    for i in range(N_QUBITS):
        qml.RX(inputs[i], wires=i)
    qml.templates.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]

class HybridQNN(nn.Module):
    def __init__(self, input_dim, n_qubits, n_layers):
        super().__init__()
        self.n_qubits = n_qubits
        self.projection = nn.Linear(input_dim, n_qubits)
        self.q_weights = nn.Parameter(
            0.01 * torch.randn(n_layers, n_qubits, 3)
        )
        self.fc = nn.Linear(n_qubits, 1)

    def forward(self, x):
        x_proj = torch.tanh(self.projection(x)) * np.pi
        q_out = []
        for sample in x_proj:
            q = qnn_layer(sample, self.q_weights)
            q = torch.stack(q)
            q_out.append(q)
        q_out = torch.stack(q_out).float()
        return self.fc(q_out)

In [ ]:
# Load Model
input_dim = X_test.shape[1]
model = HybridQNN(input_dim, N_QUBITS, N_LAYERS)

model_path = 'trained_model/hybrid_qnn_model.pth'
if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path))
    print("Model loaded successfully.")
else:
    print("Error: Model file not found!")

## 3. Evaluation

In [ ]:
model.eval()
all_preds = []
all_labels = []

print("Running inference on test set...")
with torch.no_grad():
    for X_batch, y_batch in tqdm(test_loader):
        logits = model(X_batch)
        preds = torch.sigmoid(logits) > 0.5
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y_batch.cpu().numpy())

# Metrics
acc = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds)
rec = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)

print(f"\nTest Results:")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 Score:  {f1:.4f}")

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Benign', 'Attack'], yticklabels=['Benign', 'Attack'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()